In [1]:
# imports

import os
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from PIL import Image
from rich import print


In [2]:
# Define paths

BASE_DIR = Path("../data/raw/ham10000")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = BASE_DIR / "HAM10000_metadata.csv"
assert metadata_path.exists(), "Metadata file missing!"

In [3]:
# Load metadata

df = pd.read_csv(metadata_path)
print(f"Total rows in metadata: {len(df)}")
df.head()

Total rows in metadata: 10015

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [4]:
# Attach image paths

PROJECT_ROOT = Path().resolve().parent  # Case_Study root
BASE_DIR = PROJECT_ROOT / "data/raw/ham10000"

def find_image(image_id):
    for folder in ["ham10000_images_part_1", "ham10000_images_part_2"]:
        img_path = BASE_DIR / folder / f"{image_id}.jpg"
        if img_path.exists():
            return str(img_path.resolve())  # absolute path
    return None


image_paths = []

for image_id in tqdm(df["image_id"]):
    path = find_image(image_id)
    assert path is not None, f"Missing image: {image_id}"
    image_paths.append(str(path))

df["image_path"] = image_paths

print("All images validated")

  0%|          | 0/10015 [00:00<?, ?it/s]

All images validated

In [5]:
# Extract resolution and filter out corrupted images

widths = []
heights = []

for path in tqdm(df["image_path"]):
    try:
        with Image.open(path) as img:
            w, h = img.size
        widths.append(w)
        heights.append(h)
    except:
        widths.append(None)
        heights.append(None)

df["width"] = widths
df["height"] = heights

# Drop corrupted images
before = len(df)
df = df.dropna(subset=["width", "height"])
after = len(df)

print(f"Removed corrupted images: {before - after}")


  0%|          | 0/10015 [00:00<?, ?it/s]

Removed corrupted images: 0

In [6]:
# Save clean csv

save_path = PROCESSED_DIR / "ham10000_clean.csv"
df.to_csv(save_path, index=False)
print(f"Saved cleaned dataset at {save_path}")


Saved cleaned dataset at ../data/processed/ham10000_clean.csv